# Sabak Voice Lab
تجربة أصوات للعبة عمر. شغّل الخلايا بالترتيب من فوق.
قبلها: من القائمة **Runtime > Change runtime type** اختر **T4 GPU**.

In [ ]:
# 1) تثبيت الأدوات (3 إلى 5 دقائق)
!pip -q install coqui-tts faster-whisper
import os
os.environ["COQUI_TOS_AGREED"]="1"   # ترخيص XTTS غير تجاري: للتجربة فقط
print("تم")

In [ ]:
# 2) ارفع عينة صوت (6 إلى 20 ثانية، صوت واحد نظيف بدون موسيقى) باسم أي ملف wav/mp3/m4a
from google.colab import files
up = files.upload()
ref = list(up.keys())[0]
!ffmpeg -y -loglevel error -i "{ref}" -ac 1 -ar 22050 ref.wav
print("العينة جاهزة:", ref)

In [ ]:
# 3) الجمل. غيّرها كما تريد (الفصحى البسيطة أوضح للنموذج من اللهجة)
LINES = {
 "omar_01": "ماما! ماما! أنا لبست وحدي! شوفي، حتى الحذاء!",
 "omar_02": "بابا، ليش يقولون مرض؟ مثل الحمى؟",
 "omar_03": "لا! لا يا بابا! أنا ما أبي! أنا بكون مؤدب، والله!",
 "omar_04": "سبعة... ثمانية... تسعة... عشرة... بابا؟",
 "father_01": "لا تخاف يا عمر، أنا معك. تذكّر المعلم.",
}
LANG = "ar"   # ar عربي، en إنجليزي

In [ ]:
# 4) توليد الأصوات
import torch
from TTS.api import TTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("out", exist_ok=True)
for k,t in LINES.items():
    tts.tts_to_file(text=t, speaker_wav="ref.wav", language=LANG, file_path=f"out/{k}.wav")
    print("✓", k)

In [ ]:
# 5) فحص الوضوح: Whisper يكتب ما سمعه. قارنه بالجملة الأصلية
from faster_whisper import WhisperModel
wm = WhisperModel("small", device="cuda", compute_type="float16")
for k,t in LINES.items():
    segs,_ = wm.transcribe(f"out/{k}.wav", language=LANG)
    heard = " ".join(s.text for s in segs)
    print(k, "\n  الأصل:", t, "\n  سمع  :", heard)

In [ ]:
# 6) استمع لكل ملف هنا، ثم نزّلها كلها
from IPython.display import Audio, display
for k in LINES: 
    print(k); display(Audio(f"out/{k}.wav"))
!zip -qr voices.zip out
files.download("voices.zip")